## What is Vectorization?

Vectorization means performing an operation on an entire array at once without writing explicit Python loops.

Instead of writing

    new_salary = []

    for salary in salaries:
        new_salary.append(salary + 5000)

we simply write

    new_salary = salaries + 5000

NumPy internally loops over the array in optimized C code.

### Example Dataset

In [1]:
import numpy as np

salary = np.array([45000, 55000, 60000, 72000, 85000])

print(salary)

[45000 55000 60000 72000 85000]


### Example 1: Add Bonus

Without loop

In [2]:
updated_salary = salary + 5000

print(updated_salary)

[50000 60000 65000 77000 90000]


Output

    [50000 60000 65000 77000 90000]

Notice
We never wrote

    for

### What Happens Internally?

You wrote

    salary + 5000

Internally NumPy performs

    45000 + 5000
    55000 + 5000
    60000 + 5000
    72000 + 5000
    85000 + 5000

But the loop is written in C, not Python.

### Example 2: Multiply Entire Array

In [3]:
salary * 2

array([ 90000, 110000, 120000, 144000, 170000])

### Example 3: Division

In [5]:
salary / 1000

array([45., 55., 60., 72., 85.])

Output

    [45. 55. 60. 72. 85.]

### Example 4: Subtraction

In [6]:
salary - 3000

array([42000, 52000, 57000, 69000, 82000])

### Example 5: Exponent

In [7]:
salary ** 2

array([2025000000, 3025000000, 3600000000, 5184000000, 7225000000])

Output

Large squared values.
Every element is squared.

In [4]:
print(salary)

[45000 55000 60000 72000 85000]


### Vectorization Between Two Arrays

In [8]:
basic = np.array([40000,50000,60000])

hra = np.array([8000,10000,12000])

gross = basic + hra

print(gross)

[48000 60000 72000]


In [9]:
allowance = np.array([5000,6000,7000])

gross = basic + hra + allowance

print(gross)

[53000 66000 79000]


### Comparison Operators

Vectorization also works here.

In [10]:
salary > 60000

array([False, False, False,  True,  True])

In [11]:
salary == 55000

array([False,  True, False, False, False])

### Logical Operations

In [13]:
attendance = np.array([96,82,74,91,68])

res= attendance >= 90
print(res)

[ True False False  True False]


### Chaining Operations

In [15]:
res = attendance[(attendance >= 80) & (attendance <= 95)]
print(res)

[82 91]


Output

    [82 91]

Notice
No loop.

### Example 6

Increase salary by 10%.

In [16]:
salary * 1.10

array([49500., 60500., 66000., 79200., 93500.])

## Round Values

In [17]:
np.round(salary * 1.10,2)

array([49500., 60500., 66000., 79200., 93500.])

### Python vs NumPy

Pure Python

In [18]:
salary = [45000,55000,60000]

updated=[]

for s in salary:
    updated.append(s+5000)

print(updated)

[50000, 60000, 65000]


NumPy

In [19]:
salary=np.array([45000,55000,60000])

updated=salary+5000

print(updated)

[50000 60000 65000]


### Why is NumPy Faster?

Python

    Python Loop
    ↓
    Python Interpreter
    ↓
    Object Lookup
    ↓
    Addition
    ↓
    Append
    ↓
    Repeat Millions of Times

Lots of overhead.

NumPy

    Whole Array
    ↓
    Optimized C Loop
    ↓
    SIMD Instructions
    ↓
    Continuous Memory
    ↓
    Result

Very little overhead.

### Time Comparison

In [20]:
import time

numbers = list(range(1_000_000))

start = time.time()

result=[]

for n in numbers:
    result.append(n+10)

print(time.time()-start)

0.25470852851867676


In [21]:
numbers=np.arange(1_000_000)

start=time.time()

result=numbers+10

print(time.time()-start)

0.03787875175476074


- Usually NumPy is 10–100× faster, depending on the operation and hardware.

### Real Data Engineering Example

Suppose you want to increase every employee's salary by 5%.

Python

    for employee in employees:
        employee.salary *= 1.05

NumPy

    salary = salary * 1.05

Millions of rows are processed efficiently.

### Why Data Engineers Love Vectorization

Instead of

    Loop over every row

they write

    salary = salary * 1.08
    attendance = attendance / working_days
    tax = gross * 0.12
    bonus = salary + allowance

The entire dataset is processed at once.

## Mini Challenge

In [22]:
salary=np.array([45000,55000,60000,70000,80000])

### Q1

Increase every salary by ₹3000.

In [23]:
salary + 3000

array([48000, 58000, 63000, 73000, 83000])

### Q2

Reduce every salary by 5%.

In [24]:
salary * 0.95

array([42750., 52250., 57000., 66500., 76000.])

### Q3

Convert salaries to thousands.

In [25]:
salary / 1000

array([45., 55., 60., 70., 80.])

### Q4

Find salaries greater than ₹60,000.

In [26]:
salary[salary > 60000]

array([70000, 80000])

### Q5

Calculate:

    Gross Salary =
    Basic Salary
    +
    HRA
    +
    Special Allowance

using

    basic=np.array([40000,50000,60000])
    hra=np.array([8000,10000,12000])
    special=np.array([5000,6000,7000])

without any loop.

In [27]:
basic=np.array([40000,50000,60000])

hra=np.array([8000,10000,12000])

special=np.array([5000,6000,7000])

gross_salary = basic + hra + special

print(gross_salary)

[53000 66000 79000]


## Chapter 11: Broadcasting

Now we move to another concept that makes NumPy extremely powerful.

Until now we've only added arrays having the same shape.

    salary = np.array([50000, 60000, 70000])
    bonus = np.array([5000, 5000, 5000])
    salary + bonus

Both arrays have shape

    (3,)

Easy — NumPy just adds element by element because both arrays line up perfectly, position by position.

But what if we write

    salary + 5000

Here `salary` has 3 elements, but `5000` is just a single number (a scalar) — it has no shape at all. So how can NumPy add **one** number to **three** numbers? It can't just "add" a single value to an entire array unless it somehow treats that single value as if it were repeated three times.

This is possible because of **Broadcasting**.

### What is Broadcasting?

Broadcasting is NumPy's mechanism for performing arithmetic operations between arrays of different shapes by virtually expanding the smaller array without actually copying the data.

The key word here is **virtually**. NumPy does not physically create a bigger array in memory. Instead, it uses clever internal bookkeeping (adjusting "strides" — the step sizes NumPy uses to walk through memory) so that the smaller array *behaves* as if it were repeated, while only the original, smaller data actually exists in memory. This is what makes broadcasting both fast and memory-efficient.

### Example 1

In [28]:
import numpy as np

salary = np.array([50000, 60000, 70000])

print(salary)

[50000 60000 70000]


In [29]:
salary + 5000

array([55000, 65000, 75000])

### What Happened Internally?

You wrote

    salary + 5000

Internally NumPy behaves as if

    salary
    [50000
     60000
     70000]

    +

    5000

became

    [50000 60000 70000]
    +
    [5000 5000 5000]

Result

    [55000 65000 75000]

**Important**

NumPy does **NOT** actually create

    [5000, 5000, 5000]

That would waste memory — imagine doing this on an array with a million elements; physically duplicating the scalar a million times would be wasteful and slow.

Instead, NumPy **virtually expands** it during computation. It keeps track internally that "this value has a stride of 0 in this dimension," which is a technical way of saying "keep reading the same single value again and again instead of moving to a new memory location." The math is performed as though the expansion happened, but the actual memory footprint stays tiny.

This is called **Broadcasting**.

### Example 2: Multiply Entire Array

In [30]:
salary * 1.10

array([55000., 66000., 77000.])

Output

    [55000. 66000. 77000.]

Again,

    1.10

is broadcast to

    [1.10
     1.10
     1.10]

NumPy treats the single scalar `1.10` as if it were present at every position of the array, multiplies element-wise, and returns a new array — again, without ever physically building that repeated array of `1.10`s.

### Example 3: Subtract

In [31]:
salary - 2000

array([48000, 58000, 68000])

### Broadcasting with 2D Arrays

Broadcasting isn't limited to a 1D array and a scalar — it also applies to 2D arrays. Let's see how a single number interacts with an entire matrix.

In [32]:
employees = np.array([
    [50000,8000],
    [60000,10000],
    [70000,12000]
])

print(employees)

[[50000  8000]
 [60000 10000]
 [70000 12000]]


In [33]:
print(employees.shape)

(3, 2)


Output

    (3,2)

- Suppose we want to add ₹500 to every value.

In [34]:
employees + 500

array([[50500,  8500],
       [60500, 10500],
       [70500, 12500]])

Output

    [[50500  8500]
     [60500 10500]
     [70500 12500]]

Again,

    500

is broadcast everywhere — across all 3 rows and both columns. NumPy conceptually stretches the scalar to fill the full `(3,2)` shape of the array, so every single one of the 6 elements gets 500 added to it, using only one number stored in memory.

### Broadcasting a Row Vector

Now let's go one level further: instead of broadcasting a single scalar, we broadcast an entire 1D array (a "row vector") across a 2D array.

In [35]:
employees = np.array([
    [50000,8000],
    [60000,10000],
    [70000,12000]
])

bonus = np.array([5000,1000])

print(bonus.shape)

(2,)


In [36]:
employees + bonus

array([[55000,  9000],
       [65000, 11000],
       [75000, 13000]])

Output

    [[55000  9000]
     [65000 11000]
     [75000 13000]]

Internally

    bonus
    [5000 1000]

becomes

    [5000 1000]
    [5000 1000]
    [5000 1000]

Result

    [[55000 9000]
     [65000 11000]
     [75000 13000]]

Notice what's happening: `bonus` has shape `(2,)`, matching the number of **columns** in `employees` which is `(3,2)`. So NumPy lines the `bonus` array up against each row and repeats it down all 3 rows — the first value of `bonus` (5000) always adds to column 0, and the second value (1000) always adds to column 1. This only works because the last dimension of both shapes matches (2 columns = 2 values in `bonus`).

### Broadcasting a Column Vector

We just broadcast a value across rows (same values reused down each row). Now let's broadcast a value across columns instead — this needs a **column vector**, shaped as `(3,1)` rather than a flat `(3,)`.

In [37]:
employees = np.array([
    [50000,8000],
    [60000,10000],
    [70000,12000]
])

increment = np.array([[1000],
                      [2000],
                      [3000]])

print(increment.shape)

(3, 1)


In [38]:
employees + increment

array([[51000,  9000],
       [62000, 12000],
       [73000, 15000]])

Output

    [[51000  9000]
     [62000 12000]
     [73000 15000]]

Notice

    1000
    2000
    3000

gets repeated across columns. Because `increment` has shape `(3,1)`, its single column is stretched out to match both columns of `employees`. So row 0 gets +1000 added to *both* of its values, row 1 gets +2000 added to *both* of its values, and row 2 gets +3000 added to *both* of its values. This is the opposite direction of stretching compared to the row-vector example above — here we're expanding along columns instead of rows.

### Broadcasting Rules

This is the part that tells you, in advance, whether two shapes are even allowed to be combined — and exactly how NumPy will stretch each one. Understanding this rule means you'll never have to guess whether an operation will work.

**The rule: compare shapes dimension by dimension, starting from the rightmost (last) dimension and moving left.** For each pair of dimensions being compared, they are compatible if:
- they are equal, OR
- one of them is 1 (that dimension gets stretched to match the other), OR
- one of the arrays simply doesn't have that dimension at all (it's treated as if it were 1)

If every pair of dimensions satisfies one of these conditions, broadcasting succeeds. If even one pair fails, NumPy raises an error.

Suppose

    Shape A
    (3,2)

    Shape B
    (2,)

Compare dimensions from right to left.

    (3,2)
    (  2)

Last dimension

    2 == 2
    Valid

Missing dimension becomes

    1

So

    (1,2)

becomes

    (3,2)

Broadcast succeeds.

Another example

    (3,2)
    +
    (3,1)

Compare

    2 vs 1
    Compatible

    3 vs 3
    Compatible

Works perfectly — the `1` in `(3,1)` is the "stretchable" dimension, so it expands to match the `2` on the other array, giving both arrays an effective shape of `(3,2)` during the operation.

### Invalid Broadcasting

Not every pair of shapes can be broadcast together. Here's what happens — and why — when the rule above is violated.

In [39]:
a = np.array([1,2,3])

b = np.array([10,20])

a + b

ValueError: operands could not be broadcast together with shapes (3,) (2,) 

### Why does this fail?

- Compare the shapes: `3` vs `2`
- Neither dimension is `1`
- There's no rule that lets a `3` and a `2` line up — neither one is a `1` that could stretch to match the other, and they aren't equal either
- NumPy has no way to decide "should the `2` repeat 1.5 times to become `3`?" — that's meaningless
- So it refuses and raises a `ValueError` instead of guessing
- This is exactly the safety check that protects you from silently getting wrong results from mismatched data

---

## Doubt: Why doesn't NumPy convert `(3,)` and `(2,)` into `(3,1)` and `(2,1)`?

### The Broadcasting Rule

- NumPy compares shapes from the **rightmost dimension to the left**
- It only inserts missing dimensions at the **FRONT (left side)**, never at the end
- This is the rule — memorize it

### Example 1: Where padding works

- Suppose:
  - `employees.shape = (3,2)`
  - `bonus.shape = (2,)`
- NumPy first rewrites the smaller shape:
  - `(2,)` becomes `(1,2)`
  - Notice — it became `(1,2)`, **NOT** `(2,1)`, because NumPy only inserts dimensions on the left
- Now compare, from the right:
  - `2 == 2` ✔
  - `3 vs 1` ✔ (since one side is `1`, it stretches)
- Since one dimension is `1`, `(1,2)` is stretched into `(3,2)`

### Back to your example

- `a.shape = (3,)`
- `b.shape = (2,)`
- Your question: why can't NumPy make them `(3,1)` and `(2,1)`?

**The answer:**
- Broadcasting **never invents new dimensions like that**
- NumPy only pads **missing leading dimensions**
- Both arrays already have one dimension: `(3,)` and `(2,)`
- Neither one is "missing" a dimension, so there's nothing to pad
- So NumPy compares them directly: `3` vs `2`
- Not equal, and neither is `1` → broadcasting fails

### Why doesn't NumPy just convert them to `(3,1)` and `(2,1)` anyway?

- Imagine NumPy did this:
  - `(3,)` → `(3,1)`
  - `(2,)` → `(2,1)`
- Now compare:
  - Right dimension: `1 == 1` ✔
  - Next: `3 vs 2` → still incompatible
- **It still fails** — so even this proposed conversion doesn't solve anything

### The deeper problem: there's no unique correct answer

- Suppose:
  - `a = [1,2,3]`
  - `b = [10,20]`
- What should NumPy produce? Options are all equally "valid" and equally arbitrary:
  - `11 22 13`
  - `11 12 23`
  - `11 21 12`
- There is **no mathematically correct answer**
- Should `10, 20` repeat? How many times?
  - `10 20 10 ?`
  - `10 20 20 ?`
- There is no unique interpretation — that's why NumPy refuses

### Why does `(2,) → (1,2)` work but this doesn't?

- Suppose `employees` has shape `(3,2)`:
[
salary hra
salary hra
salary hra
]

- `bonus` has shape `(2,)`, meaning:

[
salary_bonus
hra_bonus
]

- NumPy understands: **"this is one row"**
- So it becomes `(1,2)`:

salary_bonus hra_bonus

- Then it repeats that row three times:

salary_bonus hra_bonus
salary_bonus hra_bonus
salary_bonus hra_bonus

- This is **unambiguous** — there's only one sensible way to repeat a full row to match more rows

### Visual comparison

**Works:**

Employees (3,2) Bonus (2,) becomes (1,2) Repeat
A B X Y X Y X Y
C D X Y
E F X Y


**Fails:**

(3,) (2,)
1 2 3 + 10 20

- NumPy asks: should I do
  - `1+10, 2+20, 3+?`
  - or `1+10, 2+10, 3+20`
  - or `1+20, 2+10, 3+20`
- There is no correct rule, so it throws `ValueError` instead of guessing

### The Real Rule (Memorize This)

NumPy never reshapes arrays arbitrarily. It only does two things:

1. Adds missing dimensions **to the left**
2. Expands dimensions whose size is `1`

It **never**:

- Changes `2` into `3`
- Changes `(3,)` into `(3,1)` automatically
- Changes `(2,)` into `(2,1)` automatically
- Guesses how many times values should repeat

These restrictions are exactly what make broadcasting **predictable and safe**.

### Real Data Engineering Example

Suppose

    Employee Salary
    Employee Bonus

stored as

In [40]:
salary = np.array([50000,60000,70000])

bonus = 3000

In [41]:
salary + bonus

array([53000, 63000, 73000])

Instead of loop,
- NumPy broadcasts automatically.

- Another example

- Department allowances

### Why Broadcasting is Important

Without broadcasting

    Create temporary arrays
    Copy data
    Loop manually

Each of those steps costs time and memory: you'd need to manually build a full-sized array matching the bigger shape, physically copy the smaller array's values into every required position, and then loop through element by element to combine them — all before Python even performs the actual arithmetic.

With broadcasting

    Less memory
    Less code
    Higher speed
    Cleaner syntax

Because NumPy never actually creates that expanded copy — it just reuses the existing memory of the smaller array while performing the operation directly in optimized C code — you get the same result as manual looping and copying, but dramatically faster and using far less memory, all from a single line of code like `employees + allowance`.

## Chapter 12: Universal Functions (ufuncs)

### What is a Universal Function (ufunc)?

A Universal Function (ufunc) is a function that operates element-wise on one or more NumPy arrays.

Instead of writing loops, NumPy applies the function to every element automatically using optimized C code.

For example,

    np.sqrt(arr)

calculates the square root of every element in the array.

### Why are ufuncs important?

Suppose you have salaries of 10 million employees.

Without NumPy:

    result = []

    for salary in salaries:
        result.append(salary * 1.10)

With NumPy:

    result = salaries * 1.10

Similarly,

Instead of

    for x in arr:
        print(math.sqrt(x))

we simply write

    np.sqrt(arr)

The function automatically works on the whole array.

### Categories of ufuncs

NumPy provides hundreds of ufuncs.

The major categories are

    Universal Functions
    │
    ├── Arithmetic
    ├── Trigonometric
    ├── Exponential
    ├── Logarithmic
    ├── Comparison
    ├── Rounding
    └── Statistical Helpers

### Example Dataset

In [42]:
import numpy as np

salary = np.array([40000, 50000, 60000, 70000])

print(salary)

[40000 50000 60000 70000]


In [ ]:
### 1. Square Root

In [44]:
numbers = np.array([1,4,9,16,25])

print(np.sqrt(numbers))
print(numbers)

[1. 2. 3. 4. 5.]
[ 1  4  9 16 25]


Output

    [1. 2. 3. 4. 5.]

NumPy applies `sqrt()` to every element.

Internally

    sqrt(1)
    sqrt(4)
    sqrt(9)
    sqrt(16)
    sqrt(25)

### 2. Square

In [45]:
print(np.square(numbers))

[  1  16  81 256 625]


Output

    [  1  16  81 256 625]

Equivalent to

    numbers ** 2

### 3. Power

Raise every element to any power.

In [47]:
print(np.power(numbers,3))
print(numbers)

[    1    64   729  4096 15625]
[ 1  4  9 16 25]


### 4. Absolute Value

Useful in finance.

In [48]:
profit = np.array([-5000,4000,-2000,9000])

print(np.abs(profit))

[5000 4000 2000 9000]


### 5. Exponential

In [49]:
arr = np.array([1,2,3])

print(np.exp(arr))

[ 2.71828183  7.3890561  20.08553692]


Output

Approximately

    [ 2.718 7.389 20.086]

Here

    e¹
    e²
    e³

### 6. Natural Log

In [50]:
arr = np.array([1,10,100])

print(np.log(arr))

[0.         2.30258509 4.60517019]


Output

    [0.
     2.3025
     4.6051]

Natural logarithm

Base

    e

### 7. Log Base 10

In [51]:
print(np.log10(arr))

[0. 1. 2.]


### 8. Trigonometric Functions

In [52]:
angles = np.array([0,np.pi/2,np.pi])

print(np.sin(angles))

[0.0000000e+00 1.0000000e+00 1.2246468e-16]


In [53]:
print(np.cos(angles))

[ 1.000000e+00  6.123234e-17 -1.000000e+00]


In [54]:
print(np.tan(angles))

[ 0.00000000e+00  1.63312394e+16 -1.22464680e-16]


**Important**

NumPy uses radians, not degrees.

If you have degrees,
Convert first.

    angles = np.deg2rad([0,30,45,60,90])
    print(np.sin(angles))

In [55]:
angles = np.deg2rad([0,30,45,60,90])

print(np.sin(angles))

[0.         0.5        0.70710678 0.8660254  1.        ]


### 9. Maximum

In [58]:
salary = np.array([50000,60000,70000])

bonus = np.array([3000,5000,2000])

print(np.maximum(salary,bonus))

[50000 60000 70000]


Output

    [50000 60000 70000]

It compares element by element.

Another example

In [59]:
a = np.array([5,8,3])

b = np.array([7,2,9])

print(np.maximum(a,b))

[7 8 9]


### 10. Minimum

In [60]:
print(np.minimum(a,b))

[5 2 3]


### 11. Clip

Very useful.

Suppose attendance cannot exceed

    100

or go below

    0

In [61]:
attendance = np.array([-5,45,110,88])

print(np.clip(attendance,0,100))

[  0  45 100  88]


### 12. Floor

Rounds downward.

In [62]:
arr = np.array([3.9,5.1,8.7])

print(np.floor(arr))

[3. 5. 8.]


### 13. Ceil

Rounds upward.

In [63]:
print(np.ceil(arr))

[4. 6. 9.]


### 14. Round

In [64]:
arr = np.array([3.14159,5.67891])

print(np.round(arr,2))

[3.14 5.68]


### Can ufuncs Work on 2D Arrays?

Yes.

In [65]:
matrix = np.array([
    [1,4],
    [9,16]
])

print(np.sqrt(matrix))

[[1. 2.]
 [3. 4.]]


### Chaining ufuncs

In [66]:
salary = np.array([50000,60000,70000])

result = np.sqrt(
    np.round(
        salary/1000,
        2
    )
)

print(result)

[7.07106781 7.74596669 8.36660027]


## Multiple ufuncs can be chained together.

### Real Data Engineering Example

Suppose salary cannot exceed

    150000

In [68]:
salary = np.clip(salary,0,150000)
print(salary)

[50000 60000 70000]


Suppose attendance values have decimals.

In [69]:
attendance = np.round(attendance,2)
print(attendance)

[ -5  45 110  88]


Need absolute errors?

In [70]:
errors = np.abs(actual-predicted)
print(errors)

NameError: name 'actual' is not defined

Need tax calculation?

In [71]:
tax = np.round(gross_salary*0.12,2)
print(tax)

[6360. 7920. 9480.]


## Chapter 13: Mathematical Operations

This chapter is different from vectorization and ufuncs.

- **Vectorization** → How NumPy performs operations without Python loops.
- **ufuncs** → Built-in mathematical functions (`np.sqrt()`, `np.log()`, etc.).
- **Mathematical Operations** → Operations between arrays, including matrix multiplication and linear algebra.

These operations are used heavily in:

- Data Engineering
- Machine Learning
- Deep Learning
- Computer Vision
- Scientific Computing

### Types of Mathematical Operations

NumPy supports:

    Arithmetic Operations
            │
            ├── +
            ├── -
            ├── *
            ├── /
            ├── //
            ├── %
            ├── **
            │
            ▼
    Linear Algebra Operations
            │
            ├── Dot Product
            ├── Matrix Multiplication
            ├── Inner Product
            ├── Outer Product
            └── Cross Product

### Example Dataset

In [72]:
import numpy as np

basic = np.array([40000,50000,60000])

hra = np.array([8000,10000,12000])

print(basic)
print(hra)

[40000 50000 60000]
[ 8000 10000 12000]


### 1. Addition

In [73]:
gross = basic + hra

print(gross)

[48000 60000 72000]


Output

    [48000 60000 72000]

Element-wise

    40000 + 8000
    50000 + 10000
    60000 + 12000

In [74]:
### 2. Subtraction
print(basic - hra)

[32000 40000 48000]


In [75]:
### 3. Multiplication
print(basic * hra)

[320000000 500000000 720000000]


Output

    [320000000 500000000 720000000]

Notice

This is **NOT** Matrix multiplication.

This is **Element-wise multiplication**.

    40000 × 8000
    50000 × 10000
    60000 × 12000

In [76]:
### 4. Division
print(basic / hra)

[5. 5. 5.]


### 5. Floor Division

In [77]:
print(basic // hra)

[5 5 5]


Output

    [5 5 5]

Returns integers.

### 6. Modulus

In [78]:
print(basic % hra)

[0 0 0]


### 7. Power

In [79]:
print(basic ** 2)

[1600000000 2500000000 3600000000]


### Matrix Multiplication

Most beginners confuse

    *

and

    @

They are completely different.

- `*` → element-wise multiplication (each element multiplies its corresponding element)
- `@` → true matrix multiplication (rows combined with columns using the standard linear algebra rule)

### Example Matrix

In [80]:
A = np.array([
    [1,2],
    [3,4]
])

B = np.array([
    [5,6],
    [7,8]
])

### Matrix Multiplication

In [81]:
print(A @ B)

[[19 22]
 [43 50]]


Output

    [[19 22]
     [43 50]]

This is true matrix multiplication.

Equivalent

In [82]:
print(np.matmul(A,B))

[[19 22]
 [43 50]]


or

In [83]:
print(np.dot(A,B))

[[19 22]
 [43 50]]


## for 2D matrices.
### Dot Product

Most common interview question.

In [84]:
salary = np.array([2,3,4])

hours = np.array([5,6,7])

print(np.dot(salary,hours))

56


Output

    56

Calculation

    2×5
    +
    3×6
    +
    4×7
    =
    10
    +
    18
    +
    28
    =
    56

Dot product always returns a single number for two 1D arrays.

### Inner Product

In [85]:
print(np.inner(salary,hours))

56


Output

    56

For 1D arrays,

    inner == dot

### Outer Product

In [86]:
print(np.outer(salary,hours))

[[10 12 14]
 [15 18 21]
 [20 24 28]]


Output

    [[10 12 14]
     [15 18 21]
     [20 24 28]]

Every element multiplies every other element.

### Cross Product

Used in physics.

Rarely used in Data Engineering.

In [87]:
a = np.array([1,2,3])

b = np.array([4,5,6])

print(np.cross(a,b))

[-3  6 -3]


### Matrix Transpose

In [88]:
print(A)

[[1 2]
 [3 4]]


In [89]:
print(A.T)

[[1 3]
 [2 4]]


Output

    1 3
    2 4

Rows become columns.

### Matrix Determinant

In [90]:
print(np.linalg.det(A))

-2.0000000000000004


### Matrix Inverse

In [91]:
print(np.linalg.inv(A))

[[-2.   1. ]
 [ 1.5 -0.5]]


### Eigenvalues

Used in PCA.

In [92]:
values, vectors = np.linalg.eig(A)

print(values)

[-0.37228132  5.37228132]


### Matrix Rank

In [93]:
print(np.linalg.matrix_rank(A))

2


### Solve Linear Equations

Suppose

    2x + y = 5
    x + y = 3

In [94]:
A = np.array([
    [2,1],
    [1,1]
])

b = np.array([5,3])

print(np.linalg.solve(A,b))

[2. 1.]


### Real Data Engineering Example

Suppose

    Basic Salary
    HRA
    Allowance

stored separately.

Instead of looping,

In [95]:
gross = basic + hra + allowance

Suppose

    Sales
    Quantity

Need total revenue.

Suppose

    Sales
    Quantity

Need total revenue.

Suppose

Normalize feature matrix

    X @ weights

Used in every Machine Learning algorithm.

# What are Aggregations?

An aggregation combines many values into one value.

Example:

Employee salaries

    [50000, 60000, 55000, 70000]

Instead of looking at every salary, we might want:

- Total salary
- Average salary
- Highest salary
- Lowest salary

These are aggregations.

### Common Aggregation Functions

| Function | Description |
|---|---|
| `sum()` | Total |
| `mean()` | Average |
| `median()` | Middle value |
| `std()` | Standard deviation |
| `var()` | Variance |
| `min()` | Smallest value |
| `max()` | Largest value |
| `argmin()` | Index of minimum value |
| `argmax()` | Index of maximum value |
| `prod()` | Product of all values |
| `cumsum()` | Cumulative sum |
| `cumprod()` | Cumulative product |
| `any()` | At least one True |
| `all()` | All True |

### Example Dataset

In [96]:
import numpy as np

salary = np.array([50000, 60000, 55000, 70000])

print(salary)

[50000 60000 55000 70000]


### 1. Sum


In [97]:
print(np.sum(salary))

235000


### 2. Mean

In [98]:
print(np.mean(salary))

58750.0


### 3. Median

The middle value after sorting.

In [99]:
salary = np.array([70000,50000,60000,55000])

print(np.median(salary))

57500.0


Output

    57500.0

Sorted

    50000
    55000
    60000
    70000

Middle two

    55000
    60000

Average

    57500

### 4. Minimum

In [100]:
print(np.min(salary))

50000


### 5. Maximum

In [101]:
print(np.max(salary))

70000


### 6. argmin()

Returns the index, not the value.

In [102]:
print(np.argmin(salary))

1


Output

    1

Because

    salary

is

    Index
    0 → 70000
    1 → 50000
    2 → 60000
    3 → 55000

Minimum is

    50000

at index

    1

### 7. argmax()

In [103]:
print(np.argmax(salary))

0


Output

    0

Largest value

    70000

at index

    0

### 8. Product

In [104]:
numbers = np.array([2,3,4])

print(np.prod(numbers))

24


Output

    24

Calculation

    2×3×4

### 9. Cumulative Sum

Very useful.

In [105]:
sales = np.array([10,20,15,25])

print(np.cumsum(sales))

[10 30 45 70]


Output

    [10 30 45 70]

Calculation

    10
    10+20
    10+20+15
    10+20+15+25

### 10. Cumulative Product

In [106]:
numbers = np.array([2,3,4])

print(np.cumprod(numbers))

[ 2  6 24]


### 11. Standard Deviation

Measures how spread out the data is.

In [107]:
print(np.std(salary))

7395.09972887452


Example output

    7395.1

Higher standard deviation means salaries vary more.

### 12. Variance
- print(np.var(salary))

In [109]:
print(np.var(salary))

54687500.0


### 13. any()

In [110]:
attendance = np.array([True,True,False,True])

print(np.any(attendance))

True


Output

    True

At least one value is True.

Another Example

In [111]:
attendance = np.array([False,False,False])

print(np.any(attendance))

False


### 14. all()

In [112]:
attendance = np.array([True,True,True])

print(np.all(attendance))

True


Output

    True

Every value is True.

Another example

In [113]:
attendance = np.array([True,False,True])

print(np.all(attendance))

False


## The Most Important Concept: axis

This is the topic where most beginners get confused.

Let's understand it carefully.

### Example Matrix

In [114]:
attendance = np.array([
    [20,21,22],
    [18,19,20],
    [22,22,23]
])

print(attendance)

[[20 21 22]
 [18 19 20]
 [22 22 23]]


Output

    20 21 22
    18 19 20
    22 22 23

Suppose

    Rows → Employees
    Columns → Months

              Jan Feb Mar
    Emp1      20 21 22
    Emp2      18 19 20
    Emp3      22 22 23

### Sum Without axis

In [115]:
print(np.sum(attendance))

187


Output

    187

Everything is added together.

### axis = 0

In [116]:
print(np.sum(attendance, axis=0))

[60 62 65]


Output

    [60 62 65]

Calculation

    20+18+22
    21+19+22
    22+20+23

Column-wise.

Think

    ↓
    ↓
    ↓

### axis = 1

In [117]:
print(np.sum(attendance, axis=1))

[63 57 67]


Output

    [63 57 67]

Calculation

    20+21+22
    18+19+20
    22+22+23

Row-wise.

Think

    → → →

### Mean with axis

In [118]:
print(np.mean(attendance, axis=0))

[20.         20.66666667 21.66666667]


Output

    [20.0 20.67 21.67]

Average for each month.

In [119]:
print(np.mean(attendance, axis=1))

[21.         19.         22.33333333]


Output

    [21.0 19.0 22.33]

Average attendance of each employee.

## Real Data Engineering Examples

Suppose salary data

In [120]:
salary = np.array([
    [50000,52000,54000],
    [60000,62000,64000],
    [70000,71000,72000]
])

## Average salary per employee

In [121]:
np.mean(salary, axis=1)

array([52000., 62000., 71000.])

## Average salary per month

In [122]:
np.mean(salary, axis=0)

array([60000.        , 61666.66666667, 63333.33333333])

## Maximum monthly salary

In [123]:
np.max(salary, axis=0)

array([70000, 71000, 72000])

## Highest salary of each employee

In [124]:
np.max(salary, axis=1)

array([54000, 64000, 72000])

## Chapter 15: Reshaping

This chapter is used constantly in:

- Data Engineering
- Pandas
- Machine Learning
- Deep Learning
- Computer Vision

**The idea is simple:**

> Reshaping changes the *structure (shape)* of an array without changing its *data*.

### Why Do We Need Reshaping?

Suppose you receive employee salaries as a 1D array.

In [125]:
import numpy as np

salary = np.array([50000, 60000, 70000, 80000])

print(salary)

[50000 60000 70000 80000]


In [126]:
print(salary.shape)

(4,)


Output

    (4,)

But suppose another library expects data laid out like a table:

| Employee | Salary |
|---|---|
| | 50000 |
| | 60000 |
| | 70000 |
| | 80000 |

which needs shape `(4,1)` instead of `(4,)`.

Instead of recreating the array from scratch, we simply **reshape** it — same data, different structure.

### 1. `reshape()`

The most important function in this chapter.

In [128]:
arr = np.arange(12)

print(arr)
print(arr.shape)

[ 0  1  2  3  4  5  6  7  8  9 10 11]
(12,)


Output

    (12,)

Now let's convert this into **3 rows, 4 columns**.

In [130]:
matrix = arr.reshape(3,4)

print(matrix)
print(matrix.shape)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
(3, 4)


### How Does `reshape()` Actually Work?

- It does **NOT** rearrange or shuffle the values
- It simply **reads the existing values in row-major order** and fills the new shape

Original (flat):

    0 1 2 3 4 5 6 7 8 9 10 11

Reshaped into `(3,4)`:

    0 1 2 3
    4 5 6 7
    8 9 10 11

- Nothing about the underlying data changed
- **Only the structure (how it's grouped/viewed) changed**

In [133]:
arr = np.arange(8)

print(arr)
print(arr.shape)

[0 1 2 3 4 5 6 7]
(8,)


In [132]:
print(arr.reshape(2,4))

[[0 1 2 3]
 [4 5 6 7]]


Output

    [[0 1 2 3]
     [4 5 6 7]]

Reshape the same array again, this time into `(4,2)`

In [134]:
print(arr.reshape(4,2))

[[0 1]
 [2 3]
 [4 5]
 [6 7]]


### Rule of `reshape()`

> The total number of elements must remain exactly the same before and after reshaping.

Example — this array has **12 elements**:

    arr = np.arange(12)

All of these are **valid** reshapes because each pair of dimensions multiplies to 12:

- `arr.reshape(2,6)` → 2×6=12
- `arr.reshape(3,4)` → 3×4=12
- `arr.reshape(4,3)` → 4×3=12
- `arr.reshape(6,2)` → 6×2=12
- `arr.reshape(12,1)` → 12×1=12
- `arr.reshape(1,12)` → 1×12=12

### Invalid Reshape

In [135]:
arr = np.arange(12)

arr.reshape(5,3)

ValueError: cannot reshape array of size 12 into shape (5,3)

Output

    ValueError
    cannot reshape array of size 12 into shape (5,3)

**Why does this fail?**

- `5 × 3 = 15`
- But the array only has **12** values
- NumPy cannot invent 3 extra values out of nowhere — every element in the new shape must map back to a real element in the original array

### Automatic Dimension (`-1`)

One of the most useful NumPy features — let NumPy calculate a dimension for you instead of doing the math yourself.

In [138]:
arr = np.arange(12)

print(arr.reshape(3,-1))


[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]


Output

    [[ 0 1 2 3]
     [ 4 5 6 7]
     [ 8 9 10 11]]

- You said "give me 3 rows"
- NumPy calculates `12 ÷ 3 = 4` columns automatically

In [139]:
print(arr.reshape(-1,3))

[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]


Output

    [[ 0  1  2]
     [ 3  4  5]
     [ 6  7  8]
     [ 9 10 11]]

- You said "give me 4 columns"
- NumPy calculates `12 ÷ 4 = 3` rows automatically

In [140]:
print(arr.reshape(2,-1))

[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]


### Only ONE `-1` Is Allowed

- Correct: `arr.reshape(-1,4)` — one unknown dimension is fine
- **Wrong:** `arr.reshape(-1,-1)`

Output

    ValueError

**Why?**

- NumPy can only solve for **one unknown**, the same way you can't solve a single equation with two unknown variables
- If both dimensions are `-1`, NumPy has no way to figure out how to split 12 elements into two unspecified numbers

### Reshape into a Column Vector

Very common when preparing data for ML models or matrix operations.

In [141]:
salary = np.array([50000,60000,70000])

print(salary.shape)

(3,)


In [143]:
salary = salary.reshape(-1,1)

print(salary)
print(salary.shape)

[[50000]
 [60000]
 [70000]]
(3, 1)


### Reshape into a Row Vector

In [145]:
salary = salary.reshape(1,-1)

print(salary)
print(salary.shape)

[[50000 60000 70000]]
(1, 3)


### Real Data Engineering Example

Suppose you have salaries for 6 employees, flat.

In [146]:
salary = np.array([
    50000,
    60000,
    70000,
    80000,
    90000,
    100000
])

Reshape this into **2 departments × 3 employees each**

In [147]:
salary = salary.reshape(2,3)

print(salary)
print(salary.shape)

[[ 50000  60000  70000]
 [ 80000  90000 100000]]
(2, 3)


### 2. `flatten()`

- Converts any multi-dimensional array back into a **1D array**
- Always returns a **new, independent copy** of the data (changing the flattened array will NOT affect the original)

In [149]:
matrix = np.array([
    [1,2],
    [3,4]
])

print(matrix.flatten())
print(matrix.flatten().shape)

[1 2 3 4]
(4,)


Output

    [1 2 3 4]

Shape

### 3. `ravel()`

Looks similar to `flatten()` at first glance.

In [150]:
print(matrix.ravel())

[1 2 3 4]


Output

    [1 2 3 4]

**Key difference to remember for now:**

- `flatten()` → always returns a **Copy**
- `ravel()` → usually returns a **View** (shares memory with the original array)

*(The full difference between a View and a Copy — and why it matters — is covered in the next chapter, "Views vs Copies".)*

### 4. `transpose()`

Flips the array so that rows become columns (and columns become rows).

In [151]:
matrix = np.array([
    [1,2,3],
    [4,5,6]
])

print(matrix.T)

[[1 4]
 [2 5]
 [3 6]]


### 5. `expand_dims()`

Adds a new axis (dimension) to an array — useful when a function expects data in a specific number of dimensions.

In [153]:
arr = np.array([1,2,3])
print(arr.shape)

(3,)


Current shape: `(3,)`

Add an axis at position 0 (turns it into a row)

In [154]:
print(np.expand_dims(arr, axis=0))

[[1 2 3]]


Output

    [[1 2 3]]

Shape

    (1,3)

Now add an axis at position 1 instead (turns it into a column)

In [155]:
print(np.expand_dims(arr, axis=1))

[[1]
 [2]
 [3]]


### 6. `squeeze()`

The opposite of `expand_dims()` — removes any dimensions of size `1`.

In [156]:
arr = np.array([[1],[2],[3]])

print(arr.shape)

(3, 1)


In [157]:
arr = np.squeeze(arr)

print(arr)

[1 2 3]


Output

    [1 2 3]

Shape

    (3,)

## Chapter 16: Views vs Copies

### Why Do We Need This Topic?

Suppose you have an employee salary array.

    salary = np.array([50000, 60000, 70000])

Now suppose you assign it to another variable.

    new_salary = salary

**Question:** If you modify `new_salary`, should `salary` also change?

- Many beginners expect **No**
- The answer is actually **Yes**

Let's understand why.

### Memory Diagram

In [158]:
import numpy as np

salary = np.array([50000, 60000, 70000])

new_salary = salary

Memory

              salary
                 │
                 ▼
           -----------------------
           |50000|60000|70000|
           -----------------------
                 ▲
                 │
          new_salary

- There is only **one array** in memory
- Both variables simply **point to** the same array

### Example

In [159]:
salary = np.array([50000,60000,70000])

new_salary = salary

new_salary[0] = 99999

print(new_salary)
print(salary)

[99999 60000 70000]
[99999 60000 70000]


Output

    [99999 60000 70000]
    [99999 60000 70000]

Both changed.

**Why?**

- `new_salary = salary` does **NOT** create another array
- It only copies the **reference** (the "address" pointing to the array)
- This works exactly like Python lists

In [160]:
### Check Memory Address
salary = np.array([1,2,3])

new_salary = salary

print(id(salary))
print(id(new_salary))

2619580009808
2619580009808


Output

    140569320
    140569320

- Same address
- Same object

### How Do We Create a Genuinely New, Independent Array?

Use `copy()`.

In [161]:
salary = np.array([50000,60000,70000])

new_salary = salary.copy()

new_salary[0] = 99999

print(new_salary)
print(salary)

[99999 60000 70000]
[50000 60000 70000]


Output

    [99999 60000 70000]
    [50000 60000 70000]

The original did **not** change.

Memory Diagram

    salary
       │
       ▼
    -------------------------
    50000 60000 70000
    -------------------------

    new_salary
       │
       ▼
    -------------------------
    99999 60000 70000
    -------------------------

Now there are **two separate arrays** in memory.

### View

A **View** is a different concept altogether.

- A view creates **another object**
- But it **shares the same underlying data**

In [162]:
arr = np.array([1,2,3,4])

view = arr.view()

print(id(arr))
print(id(view))

2619580009904
2619580018448


Output (different IDs, e.g.)

    220430192
    220431920

- Different objects

**BUT...** modify the view:

In [163]:
view[0] = 100

print(view)
print(arr)

[100   2   3   4]
[100   2   3   4]


Output

    [100   2   3   4]
    [100   2   3   4]

The original changed too.

**Why?**

    arr object
        ↓
    Shared Memory
        ↑
    view object

- The **objects** are different
- But the **memory** is shared

### Memory Diagram (View)

    arr
     │
     ▼
    Object A
     │
     ▼
    -------------------
    100 2 3 4
    -------------------
     ▲
     │
    Object B
     ▲
    view

- Two objects
- One shared memory block

### How to Check If Something Is a View?

Use `.base`

In [164]:
arr = np.array([1,2,3])

view = arr.view()

print(view.base)

[1 2 3]


Output

    [1 2 3]

This means: **the view is built on top of another array**.

Now check a copy instead:

In [165]:
copy = arr.copy()

print(copy.base)

None


Output

    None

- `None` means no shared memory — it's a genuine, independent copy

### Slicing

Very important to remember: **most NumPy slices are Views.**

In [166]:
arr = np.array([10,20,30,40,50])

part = arr[1:4]

print(part)

[20 30 40]


In [167]:
part[0] = 999

print(part)
print(arr)

[999  30  40]
[ 10 999  30  40  50]


Output

    [999 30 40]
    [10 999 30 40 50]

The original changed.

**Why?**

- Slicing returns a **View**, not a copy

### Fancy Indexing

A different story here.

In [168]:
arr = np.array([10,20,30,40])

new = arr[[0,2]]

print(new)

[10 30]


In [169]:
new[0] = 999

print(arr)
print(new)

[10 20 30 40]
[999  30]


Output

    [10 20 30 40]
    [999 30]

The original did **NOT** change.

**Reason:**

- Fancy indexing creates a **Copy**

### Boolean Indexing

Same behavior as fancy indexing.

In [170]:
arr = np.array([10,20,30,40])

selected = arr[arr>20]

selected[0] = 999

print(arr)
print(selected)

[10 20 30 40]
[999  40]


Output

    [10 20 30 40]
    [999 40]

- The original is unchanged
- Boolean indexing returns a **Copy**

### `flatten()`

Remember from the previous chapter.

In [171]:
matrix = np.array([
    [1,2],
    [3,4]
])

flat = matrix.flatten()

flat[0] = 999

print(flat)
print(matrix)

[999   2   3   4]
[[1 2]
 [3 4]]


Output

    [999   2   3   4]
    [[1 2]
     [3 4]]

The original is unchanged, because:

    flatten()
        ↓
      Copy

### `ravel()`

In [172]:
matrix = np.array([
    [1,2],
    [3,4]
])

flat = matrix.ravel()

flat[0] = 999

print(flat)
print(matrix)

[999   2   3   4]
[[999   2]
 [  3   4]]


Output

    [999 2 3 4]
    [[999 2]
     [3 4]]

The original **changed**, because:

    ravel()
        ↓
    View (usually)

### `reshape()`

This is interesting — `reshape()` usually returns a **View** whenever possible.

In [173]:
arr = np.arange(6)

matrix = arr.reshape(2,3)

matrix[0,0] = 999

print(arr)
print(matrix)

[999   1   2   3   4   5]
[[999   1   2]
 [  3   4   5]]


Output

    [999   1   2   3   4   5]
    [[999   1   2]
     [  3   4   5]]

- Shared memory — the original was affected too

### Summary Table

| Operation | View | Copy |
|---|---|---|
| `=` assignment | ✅ | ❌ |
| `view()` | ✅ | ❌ |
| Slicing (`arr[1:4]`) | ✅ | ❌ |
| `reshape()` | Usually | Sometimes |
| `ravel()` | Usually | Sometimes |
| `copy()` | ❌ | ✅ |
| `flatten()` | ❌ | ✅ |
| Fancy Indexing | ❌ | ✅ |
| Boolean Indexing | ❌ | ✅ |

### Real Data Engineering Example

Suppose:

In [174]:
salary = np.array([50000,60000,70000,80000])

Need only the first three salaries:

In [176]:
department_salary = salary[:3]
print(department_salary)

[50000 60000 70000]


This is a **View** — it still shares memory with `salary`.

If you modify:

    department_salary[0] = 99999

the **entire original dataset changes** too.

- This can silently introduce bugs in ETL pipelines, since you might not realize you're editing shared data
- **To avoid it, always explicitly copy when you need an independent slice:**

In [178]:
department_salary = salary[:3].copy()
print(department_salary)

[50000 60000 70000]


## Chapter 17: Stacking & Splitting

This topic is about **combining arrays** and **dividing arrays**.

As a Data Engineer, this is extremely common.

Examples:

- Combine employee data from different branches
- Merge monthly sales data
- Append new records to existing datasets
- Split training and testing datasets
- Partition data for processing

### Why Do We Need Stacking?

Suppose Branch A has employee IDs.

    branch_a = np.array([1001,1002,1003])

Branch B has

    branch_b = np.array([1004,1005,1006])

How do we combine them?

NumPy provides several stacking functions for exactly this.

### 1. `concatenate()`

The most commonly used function.

In [179]:
import numpy as np

branch_a = np.array([1001,1002,1003])

branch_b = np.array([1004,1005,1006])

employees = np.concatenate((branch_a, branch_b))

print(employees)

[1001 1002 1003 1004 1005 1006]


### 2D Example

In [180]:
a = np.array([
    [1,2],
    [3,4]
])

b = np.array([
    [5,6],
    [7,8]
])

print(np.concatenate((a,b)))

[[1 2]
 [3 4]
 [5 6]
 [7 8]]


Output

    [[1 2]
     [3 4]
     [5 6]
     [7 8]]

Default is `axis = 0` — rows are added (stacked on top of each other).

### `axis = 1`

In [181]:
print(np.concatenate((a,b), axis=1))

[[1 2 5 6]
 [3 4 7 8]]


Output

    [[1 2 5 6]
     [3 4 7 8]]

Columns are added side by side.

### Visualization

Original

    A                    B
    1 2                  5 6
    3 4                  7 8

`axis=0` (stacked vertically)

    1 2
    3 4
    5 6
    7 8

`axis=1` (stacked horizontally)

    1 2 5 6
    3 4 7 8

### 2. `vstack()`

**V**ertical Stack — equivalent to `axis=0`.

In [182]:
a = np.array([1,2,3])

b = np.array([4,5,6])

print(np.vstack((a,b)))

[[1 2 3]
 [4 5 6]]


Output

    [[1 2 3]
     [4 5 6]]

Notice: 1D arrays become **rows** in the resulting 2D array.

### 3. `hstack()`

**H**orizontal Stack.

In [183]:
a = np.array([1,2,3])

b = np.array([4,5,6])

print(np.hstack((a,b)))

[1 2 3 4 5 6]


Output

    [1 2 3 4 5 6]

For 1D arrays, it simply joins them end to end.

For 2D arrays, it stacks columns side by side.

In [184]:
print(np.hstack((a.reshape(-1,1), b.reshape(-1,1))))

[[1 4]
 [2 5]
 [3 6]]


### 4. `column_stack()`

Used very frequently — this is how you combine separate 1D arrays into a table-like 2D structure.

Suppose:

- Employee IDs: `employee_id = np.array([1001,1002,1003])`
- Salary: `salary = np.array([50000,60000,70000])`

We need:

    1001 50000
    1002 60000
    1003 70000

In [185]:
employee_id = np.array([1001,1002,1003])

salary = np.array([50000,60000,70000])

print(np.column_stack((employee_id,salary)))

[[ 1001 50000]
 [ 1002 60000]
 [ 1003 70000]]


Output

    [[ 1001 50000]
     [ 1002 60000]
     [ 1003 70000]]

This is extremely common in data processing — combining separate feature/ID columns into one dataset.

### 5. `row_stack()`

Equivalent to `np.vstack()`.

In [186]:
a = np.array([1,2,3])

b = np.array([4,5,6])

print(np.row_stack((a,b)))

[[1 2 3]
 [4 5 6]]


C:\Users\manya\AppData\Local\Temp\ipykernel_14888\996925631.py:5: DeprecationWarning: `row_stack` alias is deprecated. Use `np.vstack` directly.
  print(np.row_stack((a,b)))


### 6. `stack()`

Different from `concatenate()` — this creates a **brand new axis/dimension** instead of joining along an existing one.

Suppose:

    a = np.array([1,2,3])
    b = np.array([4,5,6])

In [187]:
a = np.array([1,2,3])

b = np.array([4,5,6])

print(np.stack((a,b)))

[[1 2 3]
 [4 5 6]]


Output

    [[1 2 3]
     [4 5 6]]

Shape

In [188]:
print(np.stack((a,b)).shape)

(2, 3)


Output

    (2,3)

Unlike `concatenate()`, `stack()` creates a **new axis**.

### Difference Between `concatenate()` and `stack()`

Example:

    a = np.array([1,2,3])
    b = np.array([4,5,6])

**`concatenate()`**

    np.concatenate((a,b))

Output

    [1 2 3 4 5 6]

Shape

    (6,)

**`stack()`**

    np.stack((a,b))

Output

    [[1 2 3]
     [4 5 6]]

Shape

    (2,3)

**Key difference:**

- `concatenate()` → joins existing arrays along an **existing** axis
- `stack()` → creates a **brand new dimension**

## Splitting Arrays

Now the reverse operation — dividing one array into multiple smaller arrays.

### 7. `split()`

In [189]:
arr = np.arange(12)

print(arr)

[ 0  1  2  3  4  5  6  7  8  9 10 11]


Output

    [0 1 2 3 4 5 6 7 8 9 10 11]

Split into 3 equal parts.

In [190]:
parts = np.split(arr,3)

print(parts)

[array([0, 1, 2, 3]), array([4, 5, 6, 7]), array([ 8,  9, 10, 11])]


Output

    [array([0,1,2,3]),
     array([4,5,6,7]),
     array([8,9,10,11])]

Note: `split()` requires the array to divide **evenly** into the requested number of parts, otherwise it raises an error.

### 8. `hsplit()`

Split columns (horizontal split).

In [191]:
matrix = np.arange(12).reshape(3,4)

print(matrix)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]


Output

    [[0 1 2 3]
     [4 5 6 7]
     [8 9 10 11]]

Split into two halves along the columns.

In [192]:
left,right = np.hsplit(matrix,2)

print(left)

print(right)

[[0 1]
 [4 5]
 [8 9]]
[[ 2  3]
 [ 6  7]
 [10 11]]


Output

    [[0 1]
     [4 5]
     [8 9]]
    [[2 3]
     [6 7]
     [10 11]]

### 9. `vsplit()`

Split rows (vertical split).

In [193]:
matrix = np.arange(12).reshape(4,3)

top,bottom = np.vsplit(matrix,2)

print(top)

print(bottom)

[[0 1 2]
 [3 4 5]]
[[ 6  7  8]
 [ 9 10 11]]


### 10. `array_split()`

Unlike `split()`, this allows **unequal** splits — very useful when the array can't be divided evenly.

In [194]:
arr = np.arange(10)

parts = np.array_split(arr,3)

for part in parts:
    print(part)

[0 1 2 3]
[4 5 6]
[7 8 9]


Output

    [0 1 2 3]
    [4 5 6]
    [7 8 9]

Notice: 10 values cannot be equally divided into 3 groups. `array_split()` handles this automatically instead of throwing an error — it just makes the earlier parts slightly larger.

### Real Data Engineering Examples

Combine monthly datasets

In [196]:
jan = np.array([100,120,130])

feb = np.array([110,125,140])

sales = np.concatenate((jan,feb))
print(sales)

[100 120 130 110 125 140]


Combine employee IDs with salaries

In [198]:
employee = np.array([1001,1002])

salary = np.array([50000,60000])

report = np.column_stack((employee,salary))
print(report)

[[ 1001 50000]
 [ 1002 60000]]


Split data for batch processing

In [200]:
arr = np.arange(15)
np.array_split(arr,4)

[array([0, 1, 2, 3]),
 array([4, 5, 6, 7]),
 array([ 8,  9, 10, 11]),
 array([12, 13, 14])]

### Process one batch at a time — this pattern is common when you need to feed data into a model or pipeline in manageable chunks instead of all at once.

## Chapter 18: NumPy Random Module

This module is used to generate random numbers, random samples, and random datasets.

You have actually already used this concept indirectly when building your Employee Report Generator (using Python's `random` module). NumPy's random module is much faster and is designed for working with **arrays**.

### Why Do We Need the Random Module?

Real-world uses include:

- Generating synthetic datasets
- Sampling data
- Simulations
- Machine Learning
- Testing algorithms
- Splitting datasets
- Random initialization of weights in Deep Learning

In [201]:
import numpy as np

### 1. `np.random.rand()`

Generates random numbers between `0` and `1`.

In [202]:
print(np.random.rand())

0.7185137534381345


Output (changes every run)

    0.742318

Generate multiple values.

In [204]:
print(np.random.rand(5))
print(np.random.rand(5).shape)

[0.05760722 0.89954679 0.36837199 0.88217394 0.9664219 ]
(5,)


In [205]:
print(np.random.rand(3,4))

[[0.65035041 0.96211132 0.973218   0.52867186]
 [0.72764214 0.51301507 0.46876564 0.87735777]
 [0.77414748 0.54530388 0.27716167 0.92859521]]


### 2. `np.random.randn()`

Generates numbers from a **standard normal distribution**.

- Mean: `0`
- Standard deviation: `1`

In [206]:
print(np.random.randn(5))

[5.57698556e-04 6.22925777e-01 9.89239918e-01 1.25459365e+00
 2.30666120e-01]


Output

    [-0.62 1.35 -0.48 0.90 -2.11]

Notice — values can be:

- Negative
- Positive
- Greater than 1

Unlike `rand()`.

### Difference

**`rand()`**

    0 ≤ value < 1

**`randn()`**

    ...
    -2.5
    -1.2
    0
    0.5
    1.8
    ...

No limits.

### 3. `np.random.randint()`

Very useful — generates **integers**.

In [207]:
print(np.random.randint(1,10))

8


Output

    6

Meaning

    1 ≤ value < 10

Upper bound is **excluded**.

Generate many values.

In [208]:
print(np.random.randint(1000,1100,10))

[1073 1007 1093 1099 1055 1030 1018 1071 1002 1084]


Example Output

    [1002 1045 1088 1090 1011 1067 1054 1004 1099 1016]

This is exactly how employee IDs could be generated.

Generate a matrix.

In [209]:
print(np.random.randint(0,100,(3,4)))

[[95 79 49 39]
 [51 66 12 67]
 [87 77 48 52]]


### 4. `np.random.uniform()`

Generate floating-point numbers within a custom range.

In [210]:
print(np.random.uniform(50000,100000,5))

[78076.34466581 51609.77970143 53380.86213538 98822.86941088
 84750.3077948 ]


Example Output

    [65012.6
     89345.2
     72010.4
     95120.1
     56430.8]

Perfect for salary generation.

### 5. `np.random.choice()`

Randomly selects values from an existing array.

In [211]:
departments = np.array([
    "IT",
    "HR",
    "Finance",
    "Sales"
])

print(np.random.choice(departments))

IT


In [212]:
print(np.random.choice(departments,5))

['Finance' 'Sales' 'HR' 'Finance' 'Sales']


Output

    ['Sales'
     'IT'
     'Finance'
     'HR'
     'Sales']

Duplicates allowed by default.

### Without replacement

In [213]:
print(
    np.random.choice(
        departments,
        4,
        replace=False
    )
)

['Sales' 'IT' 'Finance' 'HR']


Output

    ['HR'
     'Sales'
     'IT'
     'Finance']

Each value appears only once, since `replace=False` prevents picking the same value twice.

### 6. `np.random.shuffle()`

Shuffles an array **in-place**.

In [214]:
employee_ids = np.arange(1001,1011)

print(employee_ids)

[1001 1002 1003 1004 1005 1006 1007 1008 1009 1010]


Output

    [1001 1002 1003 1004 1005 1006 1007 1008 1009 1010]

Shuffle

In [215]:
np.random.shuffle(employee_ids)

print(employee_ids)

[1008 1006 1002 1001 1007 1005 1003 1009 1004 1010]


Example Output

    [1006 1001 1008 1004 1010 1005 1002 1009 1003 1007]

The **original array itself changes** — nothing new is created.

### 7. `np.random.permutation()`

Looks similar to `shuffle()`, but the key difference is:

> It returns a **new** array, leaving the original untouched.

In [216]:
employee_ids = np.arange(1001,1011)

new_ids = np.random.permutation(employee_ids)

print(employee_ids)

print(new_ids)

[1001 1002 1003 1004 1005 1006 1007 1008 1009 1010]
[1006 1007 1001 1002 1005 1003 1010 1004 1009 1008]


The original remains unchanged.

### Difference

| Function | Original Modified? |
|---|---|
| `shuffle()` | ✅ Yes |
| `permutation()` | ❌ No |

### 8. `np.random.seed()`

Very important — controls **reproducibility**.

Normally:

    print(np.random.randint(1,100,5))

Every execution gives a **different** output.

Sometimes you want the **same** random numbers every run (e.g. for debugging, testing, or reproducible experiments).

In [217]:
np.random.seed(42)

print(np.random.randint(1,100,5))

[52 93 15 72 61]


Output

    [52 93 15 72 61]

Run it again — same output, always.

**Why?**

- `np.random.seed(42)` initializes the random number generator to a **fixed starting state**
- So the same "random" sequence is produced every time, given the same seed

### 9. `default_rng()`

Modern NumPy recommends using this instead of the older global `np.random` functions.

In [219]:
rng = np.random.default_rng(42)

print(rng.integers(1,100,5))

[ 9 77 65 44 43]


This is the **recommended approach** for new code, since it avoids relying on a single shared global random state and is generally safer for larger projects.

### Real Data Engineering Examples

Generate employee IDs

In [221]:
employee_ids = np.arange(1001,1101)
print(employee_ids)

[1001 1002 1003 1004 1005 1006 1007 1008 1009 1010 1011 1012 1013 1014
 1015 1016 1017 1018 1019 1020 1021 1022 1023 1024 1025 1026 1027 1028
 1029 1030 1031 1032 1033 1034 1035 1036 1037 1038 1039 1040 1041 1042
 1043 1044 1045 1046 1047 1048 1049 1050 1051 1052 1053 1054 1055 1056
 1057 1058 1059 1060 1061 1062 1063 1064 1065 1066 1067 1068 1069 1070
 1071 1072 1073 1074 1075 1076 1077 1078 1079 1080 1081 1082 1083 1084
 1085 1086 1087 1088 1089 1090 1091 1092 1093 1094 1095 1096 1097 1098
 1099 1100]


Generate salaries

In [223]:
salary = np.random.uniform(
    40000,
    120000,
    100
)
print(salary)

[ 95641.28691409  51146.51635247  88353.39034223  83187.28730413
  56244.89797878 115428.28564464  87909.23731908  95582.79464318
 110437.42712122  89948.3238507   63650.69486702  48439.54078642
  76522.76563863  57475.23497735  73320.79582963 110662.42071351
  65947.60168042  49767.03637605  68503.82704616 112546.27532366
  61770.57995077  91815.20964331  40041.63015963  68205.50850673
  64382.50065264  53172.46825144  82727.15355004  78786.39770872
  95394.88263122  61552.98670388  59530.04177982  53463.28337383
  57501.13756585  84648.16016139  72306.89368464  45191.37976872
  60313.23311475  59750.08502709  95704.34182718  96981.64719396
  51846.95439627 119819.23880392  61342.48114202 118129.19646661
  72882.96106546  42644.05863204  67605.69984213  90748.10757611
  94456.43612438  82474.76666537  75822.65316585  84231.44712571
  87415.73791035  46468.26610662  69572.35648491  59372.79506219
 104251.18051039  77624.05075568 118673.85127159  71905.95539556
 105314.54985755 103867.6

Generate attendance

In [224]:
attendance = np.random.randint(
    18,
    24,
    100
)
print(attendance)

[21 23 20 23 18 20 18 22 19 23 19 19 23 20 22 18 21 18 21 18 23 22 21 20
 18 18 21 20 20 23 23 23 23 23 23 22 20 23 20 20 19 22 23 18 21 18 22 21
 22 20 21 20 18 18 21 21 23 22 23 22 23 20 21 18 22 22 18 23 22 20 21 18
 21 22 22 18 20 19 18 19 23 19 20 23 19 23 19 20 19 19 19 18 18 18 20 23
 22 19 19 20]


Assign departments

In [226]:
departments = np.array([
    "IT",
    "HR",
    "Finance",
    "Sales"
])

dept = np.random.choice(
    departments,
    100
)
print(dept)

['IT' 'Sales' 'IT' 'IT' 'IT' 'Sales' 'IT' 'Finance' 'Sales' 'Finance' 'IT'
 'HR' 'IT' 'Finance' 'Finance' 'Sales' 'HR' 'Sales' 'Finance' 'IT' 'HR'
 'IT' 'IT' 'HR' 'Sales' 'IT' 'Finance' 'IT' 'HR' 'HR' 'HR' 'Sales' 'IT'
 'Sales' 'Sales' 'HR' 'HR' 'Sales' 'IT' 'Finance' 'Sales' 'IT' 'HR' 'HR'
 'IT' 'Finance' 'HR' 'HR' 'HR' 'IT' 'Sales' 'HR' 'Sales' 'Finance' 'HR'
 'HR' 'Finance' 'HR' 'Sales' 'IT' 'Sales' 'IT' 'IT' 'Sales' 'Sales'
 'Sales' 'Sales' 'Finance' 'HR' 'IT' 'Sales' 'Sales' 'Sales' 'Finance'
 'Sales' 'Sales' 'Sales' 'HR' 'Finance' 'Sales' 'IT' 'IT' 'IT' 'Finance'
 'Finance' 'IT' 'Sales' 'Finance' 'IT' 'IT' 'Sales' 'HR' 'Finance' 'Sales'
 'IT' 'IT' 'Sales' 'Finance' 'HR' 'HR']


Shuffle records

In [228]:
np.random.shuffle(employee_ids)
print(employee_ids)

[1056 1024 1014 1071 1092 1047 1072 1033 1042 1088 1022 1008 1030 1005
 1055 1007 1060 1036 1070 1069 1027 1015 1003 1004 1031 1057 1018 1081
 1010 1082 1037 1041 1046 1067 1039 1051 1061 1095 1091 1043 1099 1006
 1059 1049 1064 1093 1098 1089 1016 1066 1023 1094 1085 1035 1050 1090
 1052 1045 1080 1097 1002 1038 1100 1073 1075 1011 1029 1084 1021 1001
 1053 1086 1009 1054 1019 1078 1083 1076 1020 1058 1074 1012 1065 1063
 1079 1087 1017 1034 1026 1062 1040 1028 1077 1096 1032 1013 1048 1044
 1025 1068]
